# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset basic info
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"Identifier: {meta.identifier}")
print(f"License: {meta.license}")

## 2. Data Overview
Review available record sets and their IDs, as well as listed fields and columns.

In [ ]:
# List available RecordSets and their IDs (@id)
print("Available record sets and their IDs:")
for rs in dataset.metadata.record_sets:
    print(f"  - {rs.name} (@id: {rs.id})")
    print("    Fields and columns:")
    for field in rs.fields:
        print(f"      - Field: {field.name} (@id: {field.id}), Data Type: {field.data_type}")
        for column in getattr(field, 'columns', []):
            print(f"        - Column: {column.name} (@id: {column.id}), Data Type: {column.data_type}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

<sup>**Note**: Record sets, fields, and columns are always referenced by their `@id`. Use the cell above to lookup the correct `@id` values.</sup>

In [ ]:
# Get all record set @id values
record_sets = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}
for rs_id in record_sets:
    print(f"Loading data for record set {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Columns in {rs_id}: {df.columns.tolist()}")
    print(df.head(2))

# For demonstration, pick the first record set
if len(record_sets) > 0:
    main_record_set = record_sets[0]
else:
    main_record_set = None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on criteria, normalizing numeric fields, and grouping data by attributes.
<br>**All field access should use `@id` from the metadata, as listed above.**

In [ ]:
if main_record_set is not None and len(dataframes[main_record_set]) > 0:
    df = dataframes[main_record_set].copy()
    
    # Find a likely numeric field with 'coverage', 'count', 'abundance', or 'MW' in the name
    numeric_candidates = [col for col in df.columns if any(k in col.lower() for k in ['coverage', 'count', 'abundance', 'mw', 'weight', 'number', 'intensity', 'peptide'])]
    # Fallback to the first numeric/float-like column if none found
    numeric_field_id = None
    for col in numeric_candidates:
        try:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            if df[col].dtype in [float, int] or df[col].dropna().apply(lambda x: isinstance(x, (float, int))).all():
                numeric_field_id = col
                break
        except:
            continue
    if not numeric_field_id:
        # Fallback: pick first column with numeric type
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                if df[col].dtype in [float, int]:
                    numeric_field_id = col
                    break
            except:
                continue
                
    # Print out what field will be used
    print(f"Using numeric field: {numeric_field_id}")
    
    if numeric_field_id is not None:
        # Choose a reasonable threshold based on quantiles, or static value
        threshold = df[numeric_field_id].dropna().quantile(0.7) if not df[numeric_field_id].dropna().empty else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Find a likely group field (e.g. sample, experiment, accession, description, class)
        group_candidates = [col for col in df.columns if any(k in col.lower() for k in ['sample', 'class', 'group', 'type', 'accession', 'description'])]
        group_field = group_candidates[0] if len(group_candidates) > 0 else None
        if group_field is not None:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric field found for EDA.")
else:
    print("No main record set data to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields.
<br>**Please ensure that the field IDs used below correspond to the real data columns.**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set is not None and len(dataframes[main_record_set]) > 0 and numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(dataframes[main_record_set][numeric_field_id].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.show()

    # Boxplot by group_field if available
    if group_field is not None:
        plt.figure(figsize=(12,4))
        sns.boxplot(x=group_field, y=numeric_field_id, data=dataframes[main_record_set])
        plt.title(f'{numeric_field_id} by {group_field}')
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data or numeric field for visualization.")

## 6. Conclusion
In this notebook, we loaded and briefly explored the FAIR² mass spectrometry dataset using `mlcroissant`.

- The dataset exposes its structure via Croissant: all entities, record sets, fields, and columns are referenced by their `@id`.
- We've loaded the main record set and inspected the data types and possible numeric fields for downstream analysis.
- Performed some basic filtering and normalization, and visualized the distribution of a key measurement field.

This notebook can be extended for deeper analysis or integration into ML pipelines. Always use the Croissant `@id` to reference dataset elements for maximum reproducibility.